In [6]:
from pathlib import Path
import pypdf

In [10]:
PDF_PATH = Path("../data/documents/annual_report.pdf")

print(PDF_PATH)
print("Exists:", PDF_PATH.exists())

..\data\documents\annual_report.pdf
Exists: True


In [13]:
reader = pypdf.PdfReader(PDF_PATH)

print("Number of pages:", len(reader.pages))

Number of pages: 161


In [15]:
page = reader.pages[0]

text = page.extract_text()

print(text[:3000])

In [16]:
pages = []

for page_number, page in enumerate(reader.pages):

    text = page.extract_text()

    pages.append({
        "page": page_number + 1,
        "text": text
    })

print("Pages extracted:", len(pages))

Pages extracted: 161


In [18]:
print(pages[2])

{'page': 3, 'text': "PROFILE\nVISION\nMISSION\nMOTTO\nVALUES\nPERFORMANCE HIGHLIGHTS\nBOARD OF DIRECTORS\nCBE NOOR SHARI'AH ADVISORY COMMITTEE\nEXECUTIVE MANAGEMENT \nPRESIDENT’S MESSAGE\nSHARIÁ ADVISORY COMMITTEE (SAC) STATEMENT FOR THE \nFISCAL YEAR ENDED JUNE 30 2022\n1.  HIGHLIGHTS OF GLOBAL AND DOMESTIC ECONOMIES\n1.1  The Global Economy\n1.2  The Ethiopian Economy\n2  HIGHLIGHTS OF CBE’S FINANCIAL STATEMENTS\n2.1  Income Statement\n2.1.1  Income\n2.1.2  Expense\n2.1.3  Profit\n2.2  Balance Sheet\n2.2.1  Assets\n2.2.2  Liabilities\n2.2.3 Capital and Reserves\n3 HIGHLIGHTS OF NON-FINANCIAL DEVELOPMENTS  \n3.1 Customer Base Expansion and Use of Digital Channels\n3.2 Accessibility\n3.3 Human Resource Development\nCONTENTS\nii\nii\nii\nii\nii-iii\niv-v\nvi-vii\nviii\nix-xi\nXII\n1-2\n3\n3\n3\n5\n5\n5\n6\n6\n7\n7\n8\n8\n8\n8\n8\n8\nANNEX: 2022/23 AUDITOR’S REPORT AND FINANCIAL STATEMENTS\nCONTENTS OF THE ANNEX\n10-144\n145"}


In [19]:
empty_pages = []

for page in pages:

    if not page["text"] or not page["text"].strip():
        empty_pages.append(page["page"])

print("Empty pages:", empty_pages)

Empty pages: [1, 2]


### Create a DataFrame

In [20]:
import pandas as pd

df = pd.DataFrame(pages)

df.head()

,page,text
0,1,
1,2,
2,3,PROFILE\nVISION\nMISSION\nMOTTO\nVALUES\nPERFO...
3,4,PROFILE\n Â Has been serving Ethiopia since 19...
4,5,Iv) Empowerment\n Â We distinguish employees a...


In [21]:
OUTPUT_PATH = Path("../data/processed/annual_report_pages.csv")

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved to:", OUTPUT_PATH)

Saved to: ..\data\processed\annual_report_pages.csv


### Step 4 — Document Processing & Chunking

In [22]:
from pathlib import Path
import pypdf

PDF_PATH = Path("../data/documents/annual_report.pdf")

reader = pypdf.PdfReader(PDF_PATH)

print("Number of pages:", len(reader.pages))

Number of pages: 161


In [24]:
pages = []

for page_number, page in enumerate(reader.pages):

    text = page.extract_text()

    pages.append({
        "page": page_number + 1,
        "text": text or ""
    })

print("Pages extracted:", len(pages))

Pages extracted: 161


In [25]:
print(pages[0]["text"][:3000])

In [26]:
print(pages[9]["text"][:3000])

CBE NOOR SHARI’AH ADVISORY COMMITTEE
Sheikh Mohammed Hamidin Abdulsemed (Ph.D)
Chairperson  
Mahamedzen Nur Hussen (Ph.D)
Member
Nur Abdi Gudle (Asso. Professor) 
Member
Mohammed Habib Mohammed 
(Asst. Professor)
Member
Jeilan Geleta Mame (Ph.D)
Member
viii


#### Create a cleaning function:

In [27]:
import re

def clean_text(text):
    
    # Replace multiple spaces/newlines with one space
    text = re.sub(r"\s+", " ", text)
    
    # Remove leading/trailing spaces
    text = text.strip()
    
    return text

In [28]:
sample = """
Revenue


increased     significantly

during 2025.
"""

print(clean_text(sample))

Revenue increased significantly during 2025.


In [29]:
# Clean every page

for page in pages:
    page["text"] = clean_text(page["text"])

In [31]:
print(pages[3]["text"][:2000])

PROFILE Â Has been serving Ethiopia since 1942. Â Pioneered ATM services in Ethiopia. Â Introduced Western Union Money Transfer Services to Ethiopia. Â Is the first Ethiopian bank to offer interest-free banking services. Â Has been playing a catalytic role in the socio-economic development of the nation. Â Had 1,942 branches across the country as of 30 June 2024. Â Has strong correspondent relationships with about 38 renowned foreign banks and SWIFT bilateral key arrangements with over 750 banks. Â Achieved a strong asset position of Birr 1.4 trillion as of 30 June 2024. Â Combines a wide capital base with above 48 thousand permanent and 33 thousand temporary employees. Â Is headquartered in a modern skyscraper (4B+G+48) that is the tallest in East Africa. VISION To become a world-class commercial bank financially driving Ethiopia’s future. MISSION We are committed to realizing stakeholders’ values through enhanced financial intermediation globally, deploying highly motivated and skill

### Remove empty pages

In [32]:
pages = [
    page for page in pages
    if page["text"]
]

print("Non-empty pages:", len(pages))

Non-empty pages: 159


## Our first simple chunker

In [33]:
def create_chunks(text, chunk_size=1500, overlap=200):
    
    chunks = []
    
    start = 0
    
    while start < len(text):
        
        end = start + chunk_size
        
        chunk = text[start:end]
        
        chunks.append(chunk)
        
        start += chunk_size - overlap
    
    return chunks

In [37]:
text = pages[0]["text"]

chunks = create_chunks(
    text,
    chunk_size=1500,
    overlap=200
)

print("Number of chunks:", len(chunks))

Number of chunks: 1


In [38]:
print(chunks[0])

PROFILE VISION MISSION MOTTO VALUES PERFORMANCE HIGHLIGHTS BOARD OF DIRECTORS CBE NOOR SHARI'AH ADVISORY COMMITTEE EXECUTIVE MANAGEMENT PRESIDENT’S MESSAGE SHARIÁ ADVISORY COMMITTEE (SAC) STATEMENT FOR THE FISCAL YEAR ENDED JUNE 30 2022 1. HIGHLIGHTS OF GLOBAL AND DOMESTIC ECONOMIES 1.1 The Global Economy 1.2 The Ethiopian Economy 2 HIGHLIGHTS OF CBE’S FINANCIAL STATEMENTS 2.1 Income Statement 2.1.1 Income 2.1.2 Expense 2.1.3 Profit 2.2 Balance Sheet 2.2.1 Assets 2.2.2 Liabilities 2.2.3 Capital and Reserves 3 HIGHLIGHTS OF NON-FINANCIAL DEVELOPMENTS 3.1 Customer Base Expansion and Use of Digital Channels 3.2 Accessibility 3.3 Human Resource Development CONTENTS ii ii ii ii ii-iii iv-v vi-vii viii ix-xi XII 1-2 3 3 3 5 5 5 6 6 7 7 8 8 8 8 8 8 ANNEX: 2022/23 AUDITOR’S REPORT AND FINANCIAL STATEMENTS CONTENTS OF THE ANNEX 10-144 145


In [39]:
print("\n--- SECOND CHUNK ---\n")

print(chunks[1] if len(chunks) > 1 else "No second chunk")


--- SECOND CHUNK ---

No second chunk


In [40]:
all_chunks = []

for page in pages:
    
    chunks = create_chunks(
        page["text"],
        chunk_size=1500,
        overlap=200
    )
    
    for chunk_number, chunk in enumerate(chunks):
        
        all_chunks.append({
            "page": page["page"],
            "chunk": chunk_number + 1,
            "text": chunk
        })

In [41]:
print("Total chunks:", len(all_chunks))

Total chunks: 336


In [42]:
print(all_chunks[0])

{'page': 3, 'chunk': 1, 'text': "PROFILE VISION MISSION MOTTO VALUES PERFORMANCE HIGHLIGHTS BOARD OF DIRECTORS CBE NOOR SHARI'AH ADVISORY COMMITTEE EXECUTIVE MANAGEMENT PRESIDENT’S MESSAGE SHARIÁ ADVISORY COMMITTEE (SAC) STATEMENT FOR THE FISCAL YEAR ENDED JUNE 30 2022 1. HIGHLIGHTS OF GLOBAL AND DOMESTIC ECONOMIES 1.1 The Global Economy 1.2 The Ethiopian Economy 2 HIGHLIGHTS OF CBE’S FINANCIAL STATEMENTS 2.1 Income Statement 2.1.1 Income 2.1.2 Expense 2.1.3 Profit 2.2 Balance Sheet 2.2.1 Assets 2.2.2 Liabilities 2.2.3 Capital and Reserves 3 HIGHLIGHTS OF NON-FINANCIAL DEVELOPMENTS 3.1 Customer Base Expansion and Use of Digital Channels 3.2 Accessibility 3.3 Human Resource Development CONTENTS ii ii ii ii ii-iii iv-v vi-vii viii ix-xi XII 1-2 3 3 3 5 5 5 6 6 7 7 8 8 8 8 8 8 ANNEX: 2022/23 AUDITOR’S REPORT AND FINANCIAL STATEMENTS CONTENTS OF THE ANNEX 10-144 145"}


### Put chunks into a DataFrame

In [43]:
import pandas as pd

chunks_df = pd.DataFrame(all_chunks)

chunks_df.head()

,page,chunk,text
0,3,1,PROFILE VISION MISSION MOTTO VALUES PERFORMANC...
1,4,1,PROFILE Â Has been serving Ethiopia since 1942...
2,4,2,e Â We are committed to maintaining the highes...
3,5,1,Iv) Empowerment Â We distinguish employees as ...
4,5,2,ential. Â We are committed to address the need...


In [44]:
OUTPUT_PATH = Path("../data/processed/annual_report_chunks.csv")

chunks_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved:", OUTPUT_PATH)

Saved: ..\data\processed\annual_report_chunks.csv
